### Cell 1 — Setup (thay thế toàn bộ)

In [0]:
# Cell 1 — Setup (thay thế toàn bộ)
import subprocess
subprocess.run(["pip", "install", "boto3", "-q"])

import boto3
import pandas as pd
import io
from functools import reduce
from pyspark.sql import DataFrame

AWS_ACCESS_KEY = ""
AWS_SECRET_KEY = ""
BUCKET_NAME    = ""
REGION         = "eu-north-1"

s3 = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=REGION
)

spark.sql("CREATE DATABASE IF NOT EXISTS tourgo")
spark.sql("USE tourgo")

def read_csv_from_s3(table_name):
    """Đọc tất cả timestamp subfolders, union + dedup"""
    prefix = f"raw/{table_name}/"
    response = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=prefix)

    csv_keys = []
    if 'Contents' in response:
        csv_keys = [obj['Key'] for obj in response['Contents']
                    if obj['Key'].endswith('/data.csv')]

    # Fallback về format cũ nếu chưa có subfolder
    if not csv_keys:
        fallback_key = f"raw/{table_name}/{table_name}.csv"
        obj = s3.get_object(Bucket=BUCKET_NAME, Key=fallback_key)
        content = obj['Body'].read().decode('utf-8-sig')
        pdf = pd.read_csv(io.StringIO(content))
        print(f"   [FALLBACK] {fallback_key} ({len(pdf)} rows)")
        return spark.createDataFrame(pdf)

    dfs = []
    for key in sorted(csv_keys):
        obj = s3.get_object(Bucket=BUCKET_NAME, Key=key)
        content = obj['Body'].read().decode('utf-8-sig')
        pdf = pd.read_csv(io.StringIO(content))
        dfs.append(spark.createDataFrame(pdf))

    df_union = reduce(DataFrame.union, dfs)
    return df_union.dropDuplicates(["id"])

print("Kết nối S3 & Database tourgo sẵn sàng")

Kết nối S3 & Database tourgo sẵn sàng


### Cell 2 — Bronze MERGE INTO (sửa lỗi forPath -> forName)

In [0]:
# Cell 2 — Bronze MERGE INTO (sửa lỗi forPath -> forName)
from delta.tables import DeltaTable

TABLES = ['users', 'tours', 'bookings', 'payments', 'revenues', 'reviews']
bronze_summary = {}

def ingest_bronze_merge(table_name):
    print(f"\n{'='*50}")
    print(f"ĐANG INGEST: {table_name.upper()}")

    try:
        df_new = read_csv_from_s3(table_name)
        new_count = df_new.count()
        managed_table = f"bronze_{table_name}"

        table_exists = spark.catalog.tableExists(managed_table)

        if not table_exists:
            df_new.write \
                .format("delta") \
                .mode("overwrite") \
                .option("overwriteSchema", "true") \
                .saveAsTable(managed_table)
            total = new_count
            print(f"   [INITIAL LOAD] {managed_table}: {total:,} records")
        else:
            # Dùng forName thay vì forPath — đúng với Managed Tables
            dt = DeltaTable.forName(spark, f"tourgo.{managed_table}")
            dt.alias("target").merge(
                df_new.alias("source"),
                "target.id = source.id"
            ).whenMatchedUpdateAll() \
             .whenNotMatchedInsertAll() \
             .execute()

            total = spark.read.table(managed_table).count()
            print(f"   [MERGE INTO] {managed_table}: {new_count} new/updated → Total: {total:,}")

        return {"status": "--OK--", "count": total}

    except Exception as e:
        print(f"   --LỖI--: {e}")
        return {"status": "--LỖI--", "error": str(e)}

for table in TABLES:
    bronze_summary[table] = ingest_bronze_merge(table)


ĐANG INGEST: USERS
   [FALLBACK] raw/users/users.csv (218 rows)
   [MERGE INTO] bronze_users: 218 new/updated → Total: 227

ĐANG INGEST: TOURS
   [FALLBACK] raw/tours/tours.csv (772 rows)
   [MERGE INTO] bronze_tours: 772 new/updated → Total: 779

ĐANG INGEST: BOOKINGS
   [FALLBACK] raw/bookings/bookings.csv (1321 rows)
   [MERGE INTO] bronze_bookings: 1321 new/updated → Total: 1,326

ĐANG INGEST: PAYMENTS
   [FALLBACK] raw/payments/payments.csv (794 rows)
   [MERGE INTO] bronze_payments: 794 new/updated → Total: 794

ĐANG INGEST: REVENUES
   [FALLBACK] raw/revenues/revenues.csv (788 rows)
   [MERGE INTO] bronze_revenues: 788 new/updated → Total: 788

ĐANG INGEST: REVIEWS
   [FALLBACK] raw/reviews/reviews.csv (271 rows)
   [MERGE INTO] bronze_reviews: 271 new/updated → Total: 272


### Cell 3 - BRONZE SUMMARY — In ra để chụp screenshot

In [0]:
# ============================================================
#  BRONZE SUMMARY — In ra để chụp screenshot

print("\n" + "="*55)
print("BRONZE INGESTION SUMMARY")
print("="*55)
print(f"{'Table':<12} | {'Status':<8} | {'Records':>8}")
print("-"*35)

total_ok = 0
for t, r in bronze_summary.items():
    if r["status"] == "--OK--":
        print(f"{t:<12} | {'OK':<8} | {r['count']:>8,}")
        total_ok += 1
    else:
        print(f"{t:<12} | {'LỖI':<8} | {'N/A':>8}  ← {r.get('error','')[:40]}")

print("-"*35)
print(f"{'TỔNG':<12} | {total_ok}/6 OK")
print("="*55)

# Verify đọc lại từ Delta để chắc chắn đã lưu được
print("\n VERIFY — Đọc lại từ Bronze Delta Tables:")
for t in TABLES:
    try:
        table_name = f"bronze_{t}"
        count = spark.read.table(table_name).count()
        print(f"   Bảng {table_name:<15} → {count:,} records --OK--")
    except Exception as e:
        print(f"   Bảng bronze_{t:<12} → --LỖI-- {e}")



BRONZE INGESTION SUMMARY
Table        | Status   |  Records
-----------------------------------
users        | OK       |      227
tours        | OK       |      779
bookings     | OK       |    1,326
payments     | OK       |      794
revenues     | OK       |      788
reviews      | OK       |      272
-----------------------------------
TỔNG         | 6/6 OK

 VERIFY — Đọc lại từ Bronze Delta Tables:
   Bảng bronze_users    → 227 records --OK--
   Bảng bronze_tours    → 779 records --OK--
   Bảng bronze_bookings → 1,326 records --OK--
   Bảng bronze_payments → 794 records --OK--
   Bảng bronze_revenues → 788 records --OK--
   Bảng bronze_reviews  → 272 records --OK--


### Cell 4 - PREVIEW — Xem 2 dòng đầu từng Bronze table

In [0]:
# ============================================================
#  PREVIEW — Xem 2 dòng đầu từng Bronze table
# ============================================================

for t in TABLES:
    table_name = f"bronze_{t}"
    print(f"\n{'='*50}")
    print(f" BRONZE PREVIEW: {table_name.upper()}")
    try:
        df = spark.read.table(table_name)
        df.printSchema()
        df.show(2, truncate=False)
    except Exception as e:
        print(f"--LỖI-- Không đọc được: {e}")



 BRONZE PREVIEW: BRONZE_USERS
root
 |-- id: long (nullable = true)
 |-- role: string (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- date_joined: string (nullable = true)

+---+--------+---------+--------------------------------+
|id |role    |is_active|date_joined                     |
+---+--------+---------+--------------------------------+
|1  |CUSTOMER|true     |2026-05-20 12:53:18.887000+00:00|
|2  |CUSTOMER|false    |2026-05-21 17:21:52.753000+00:00|
+---+--------+---------+--------------------------------+
only showing top 2 rows

 BRONZE PREVIEW: BRONZE_TOURS
root
 |-- id: long (nullable = true)
 |-- creator_id: long (nullable = true)
 |-- title: string (nullable = true)
 |-- price: double (nullable = true)
 |-- departure_date: string (nullable = true)
 |-- slots: long (nullable = true)
 |-- status: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- category_names: string (nullable = true)

+---+----------+-------+-----+--------------+---